# 07. Variance Decomposition

This notebook reruns the Qwen-0.5B `Regime ? Correctness` ANOVA and visualises the peak-layer variance allocation for a core ten-metric panel.
        


In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

ROOT = Path.cwd().resolve()
if not (ROOT / 'data').exists():
    ROOT = ROOT.parent
DATA = ROOT / 'data'
FIGURES = ROOT / 'figures'
FIGURES.mkdir(exist_ok=True)
sys.path.insert(0, str(ROOT))

from src.statistical_analysis import cohens_d, two_way_anova, stratified_logistic_auc

sns.set_theme(style='whitegrid', font_scale=1.1)
pd.set_option('display.max_columns', 120)

metrics = [
    'speed', 'radius_of_gyration', 'effective_dim', 'dir_consistency', 'tortuosity',
    'msd_exponent', 'cos_to_late_window', 'cos_to_running_mean', 'stabilization', 'dist_slope'
]
df = pd.read_csv(DATA / 'qwen05b' / 'unified_metrics.csv')
        


In [ ]:
rows = []
for metric in metrics:
    for layer, layer_df in df.groupby('layer'):
        result = two_way_anova(layer_df, metric)
        if result is None:
            continue
        result['metric'] = metric
        result['layer'] = int(layer)
        rows.append(result)

anova_df = pd.DataFrame(rows)
peak_rows = []
for metric, metric_df in anova_df.groupby('metric'):
    explained = metric_df['eta_sq_regime'] + metric_df['eta_sq_quality'] + metric_df['eta_sq_interaction']
    peak_rows.append(metric_df.iloc[explained.argmax()])
peak_df = pd.DataFrame(peak_rows).sort_values('eta_sq_regime', ascending=False)
peak_df[['metric', 'layer', 'eta_sq_regime', 'eta_sq_quality', 'eta_sq_interaction']]
        


In [ ]:
plot_df = peak_df[['metric', 'eta_sq_regime', 'eta_sq_quality', 'eta_sq_interaction']].copy()
plot_df['residual'] = 1 - plot_df[['eta_sq_regime', 'eta_sq_quality', 'eta_sq_interaction']].sum(axis=1)
plot_df = plot_df.set_index('metric')
plot_df[['eta_sq_regime', 'eta_sq_quality', 'eta_sq_interaction', 'residual']].plot(
    kind='bar', stacked=True, figsize=(12, 5), color=['#1f4e79', '#c0392b', '#f39c12', '#d5d8dc']
)
plt.ylabel('Variance proportion')
plt.title('Peak-layer variance decomposition')
plt.tight_layout()
plt.savefig(FIGURES / 'repro_07_variance_decomposition.png', dpi=300)
plt.show()

summary_c = anova_df[
    (anova_df['metric'].isin(['speed', 'radius_of_gyration', 'effective_dim', 'tortuosity', 'dir_consistency', 'msd_exponent']))
    & (anova_df['layer'].between(10, 16))
]['proportion_regime_of_explained'].mean()
print({'mean_peak_layer_regime_eta_sq': float(anova_df[anova_df['layer'].between(10, 16)]['eta_sq_regime'].mean()), 'summary_c_regime_share': float(summary_c)})
        
